In [1]:
from pathlib import Path
import polars as pl
import pandas as pd
from scipy import stats
from matplotlib import pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import statsmodels.api as sm
import patsy
from statsmodels.stats import multitest
import numpy as np
from joblib import Parallel, delayed
from sklearn.cross_decomposition import CCA
from cogmood_analysis.nonparam import run_reg_perms, run_reg_boots
from cogmood_analysis.permcca import permcca, semiortho, center, seber_cca
pl.Config.set_tbl_rows(100) 
pl.Config.set_tbl_cols(15) 


polars.config.Config

In [2]:
task_dir = Path('../data/task/')
survey_dat = pl.read_csv('../data/survey/survey_responses.csv')
survey_dat = survey_dat.filter(~pl.col('sex_at_birth__prefer_not_to_answer'))
survey_dat = survey_dat.with_columns(
    age2 = pl.col('age')**2,
    sex = pl.col('sex_at_birth__female'),
    age_sex = pl.col('sex_at_birth__female') * pl.col('age'),
    age2_sex = pl.col('sex_at_birth__female') * pl.col('age')**2,
    backfilled_prolific_screen_group = pl.col('prolific_screen_group').fill_null(pl.col('screen_group'))
)
survey_dat = survey_dat.filter(pl.col('backfilled_prolific_screen_group') != 'othermh')

In [3]:
exploratory_dir = Path('../data/exploratory')

In [4]:
# load and merge task data
bart_dat = pl.read_csv(task_dir / 'bart_results.csv').with_columns(
    sub_id=pl.col('subject'),
    bart__alpha=pl.col('map__alpha'),
    bart__gamma_neg=pl.col('map__gamma_neg'),
    bart__gamma_pos=pl.col('map__gamma_pos'),
    bart__tau=pl.col('map__tau'),
    bart__theta=pl.col('map__theta'),
    bart__sub_score=pl.col('sub_score'),
    bart__map_dif = pl.col('sub_score') - pl.col('map_mean'),
    bart__abs_map_dif = (pl.col('sub_score') - pl.col('map_mean')).abs(),
    bart__max_rhat=pl.col('max_rhat'),
)
bd_to_merge = bart_dat.select([
    'sub_id',
    'bart__alpha',
    'bart__gamma_neg',
    'bart__gamma_pos',
    'bart__tau',
    'bart__theta',
    'bart__sub_score',
    'bart__map_dif',
    'bart__abs_map_dif',
    'bart__max_rhat'
])
dat = survey_dat.join(bd_to_merge, on='sub_id', how='inner')
rdm_dat = pl.read_csv(task_dir / 'rdm_results.csv').with_columns(
    sub_id=pl.col('subject'),
    rdm__v0=pl.col('map__v0'),
    rdm__w_d=pl.col('map__w_d'),
    rdm__w_s=pl.col('map__w_s'),
    rdm__t0=pl.col('map__t0'),
    rdm__sigma=pl.col('map__sigma'),
    rdm__sigma_timer=pl.col('map__sigma_timer'),
    rdm__v_timer=pl.col('map__v_timer'),
    rdm__sub_score=pl.col('sub_score'),
    rdm__map_dif = pl.col('sub_score') - pl.col('map_mean'),
    rdm__abs_map_dif = (pl.col('sub_score') - pl.col('map_mean')).abs(),
    rdm__max_rhat=pl.col('max_rhat'),

)
rd_to_merge = rdm_dat.select([
    'sub_id',
    'rdm__v0',
    'rdm__w_d',
    'rdm__w_s',
    'rdm__t0',
    'rdm__sigma',
    'rdm__sigma_timer',
    'rdm__v_timer',
    'rdm__sub_score',
    'rdm__map_dif',
    'rdm__abs_map_dif',
    'rdm__max_rhat'
])
dat = dat.join(rd_to_merge, on='sub_id', how='inner')
cab_dat = pl.read_csv(task_dir / 'cab_results.csv').with_columns(
    sub_id=pl.col('subject'),
    cab__a=pl.col('map__a'),
    cab__alpha=pl.col('map__alpha'),
    cab__gamma=pl.col('map__gamma'),
    cab__kappa=pl.col('map__kappa'),
    cab__lam=pl.col('map__lam'),
    cab__nu=pl.col('map__nu'),
    cab__rho=pl.col('map__rho'),
    cab__t0=pl.col('map__t0'),
    cab__tau=pl.col('map__tau'),
    cab__w=pl.col('map__w'),
    cab__sub_score=pl.col('sub_score'),
    cab__map_dif = pl.col('sub_score') - pl.col('map_mean'),
    cab__abs_map_dif = (pl.col('sub_score') - pl.col('map_mean')).abs(),
    cab__max_rhat=pl.col('max_rhat'),
)
cd_to_merge = cab_dat.select([
    'sub_id',
    'cab__a',
    'cab__alpha',
    'cab__gamma',
    'cab__kappa',
    'cab__lam',
    'cab__nu',
    'cab__rho',
    'cab__t0',
    'cab__tau',
    'cab__w',
    'cab__sub_score',
    'cab__map_dif',
    'cab__abs_map_dif',
    'cab__max_rhat'
])
dat = dat.join(cd_to_merge, on='sub_id', how='inner')
flkr_dat = pl.read_csv(task_dir / 'flkr_results.csv')
flkr_dat = survey_dat.join(flkr_dat, on='sub_id', how='inner')
flkr_dat = flkr_dat.with_columns(
    flkr__r=pl.col('map__r'),
    flkr__p=pl.col('map__p'),
    flkr__sd0=pl.col('map__sd0'),
    flkr__K=pl.col('map__K'),
    flkr__L=pl.col('map__L'),
    flkr__thresh=pl.col('map__thresh'),
    flkr__alpha=pl.col('map__alpha'),
    flkr__t0=pl.col('map__t0'),
    flkr__map_dif = pl.col('total') - pl.col('total_map'),
    flkr__abs_map_dif = (pl.col('total') - pl.col('total_map')).abs(),
    flkr__sub_score = pl.col('total'),
    flkr__max_rhat = pl.col('max_rhat')

)
fd_to_merge = flkr_dat.select([
    'sub_id',
    'flkr__r',
    'flkr__p',
    'flkr__sd0',
    'flkr__K',
    'flkr__L',
    'flkr__thresh',
    'flkr__alpha',
    'flkr__t0',
    'flkr__sub_score',
    'flkr__map_dif',
    'flkr__abs_map_dif',
    'flkr__max_rhat'
])
dat = dat.join(fd_to_merge, on='sub_id', how='inner')
dat = dat.with_columns(
    sex_screen_group = pl.col('sex') + '__' + pl.col('backfilled_prolific_screen_group')
) 

In [5]:
# only run once to create groups, if you've got to update the files, load data to pull ids

# train_dat = dat.filter(
#     pl.int_range(pl.len()).shuffle(seed=908324234).over("sex_screen_group")
#      < (pl.len() * 0.5).over("sex_screen_group")
# )
# test_dat = dat.join(train_dat.select('sub_id'), on='sub_id', how='anti')

In [6]:
old_train_dat = pl.read_csv(exploratory_dir / 'training_data.csv')
old_test_dat = pl.read_csv(exploratory_dir / 'test_data.csv')

In [7]:
train_dat = dat.filter(pl.col('sub_id').is_in(old_train_dat.select('sub_id').to_numpy().squeeze()))
test_dat = dat.filter(pl.col('sub_id').is_in(old_test_dat.select('sub_id').to_numpy().squeeze()))

In [8]:
# make sure the split is good
assert len(test_dat.filter(pl.col('sub_id').is_in(train_dat.select('sub_id').to_numpy().squeeze()))) == 0
assert len(train_dat.filter(pl.col('sub_id').is_in(test_dat.select('sub_id').to_numpy().squeeze()))) == 0
# make sure no one is left out
assert len(dat.filter(pl.col('sub_id').is_in(train_dat.select('sub_id').to_numpy().squeeze()) | pl.col('sub_id').is_in(test_dat.select('sub_id').to_numpy().squeeze()) )) == len(dat)

In [9]:
train_dat.group_by('sex_screen_group').len().sort('sex_screen_group')

sex_screen_group,len
str,u32
"""false__anx""",114
"""false__anx_atn""",36
"""false__atn""",17
"""false__dep""",30
"""false__dep_anx""",123
"""false__dep_anx_atn""",110
"""false__dep_atn""",9
"""false__hv""",168
"""true__anx""",163


In [10]:
test_dat.group_by('sex_screen_group').len().sort('sex_screen_group')

sex_screen_group,len
str,u32
"""false__anx""",114
"""false__anx_atn""",35
"""false__atn""",16
"""false__dep""",30
"""false__dep_anx""",123
"""false__dep_anx_atn""",109
"""false__dep_atn""",9
"""false__hv""",167
"""true__anx""",162


In [12]:
train_dat.write_csv(exploratory_dir / 'training_data.csv')
test_dat.write_csv(exploratory_dir / 'test_data.csv')
train_dat.drop('survey_date').write_csv(exploratory_dir / 'public_training_data.csv')
test_dat.drop('survey_date').write_csv(exploratory_dir / 'public_test_data.csv')